# KOSIS API 데이터 탐색 노트북

KOSIS 통계목록 API를 단계별로 탐색해 **어떤 통계표를 API로 가져올 수 있는지** 확인합니다. 통계표를 고른 뒤에는 실제 통계자료 API 요청도 시험할 수 있습니다.

- API 키가 있으면 프로젝트 루트의 `.env`에 `KOSIS_API_KEY=발급받은키`를 저장하세요.
- API 키가 없어도 `output/`의 기존 JSON 샘플로 응답 구조를 살펴볼 수 있습니다.
- API 키는 노트북 출력에 표시하지 않습니다.

In [1]:
from __future__ import annotations

import json
import os
from pathlib import Path
from typing import Any
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

try:
    import pandas as pd
except ImportError:
    pd = None

ROOT = Path.cwd()
LIST_API_URL = "https://kosis.kr/openapi/statisticsList.do"
DATA_API_URL = "https://kosis.kr/openapi/Param/statisticsParameterData.do"

## 1. API 키 불러오기

In [4]:
def load_dotenv(path: Path = ROOT / ".env") -> None:
    if not path.exists():
        return
    for raw_line in path.read_text(encoding="utf-8-sig").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip("\"'"))

load_dotenv()
API_KEY = os.getenv("KOSIS_API_KEY", "").strip()
print("KOSIS API 키를 불러왔습니다." if API_KEY else "API 키가 없습니다. .env를 설정하면 실시간 조회를 사용할 수 있습니다.")

KOSIS API 키를 불러왔습니다.


## 2. 공통 요청·표시 함수

In [5]:
def request_json(url: str, params: dict[str, Any]) -> list[dict[str, Any]]:
    if not API_KEY:
        raise RuntimeError("KOSIS_API_KEY가 설정되지 않았습니다.")
    query = {**params, "apiKey": API_KEY, "format": "json", "jsonVD": "Y"}
    request = Request(
        f"{url}?{urlencode(query)}",
        headers={"User-Agent": "kosis-notebook-explorer/1.0"},
    )
    try:
        with urlopen(request, timeout=30) as response:
            charset = response.headers.get_content_charset() or "utf-8"
            payload = json.loads(response.read().decode(charset))
    except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as exc:
        raise RuntimeError(f"KOSIS 요청 실패: {exc}") from exc
    if isinstance(payload, dict):
        message = payload.get("errMsg") or payload.get("message") or payload
        raise RuntimeError(f"KOSIS API 오류: {message}")
    if not isinstance(payload, list):
        raise RuntimeError(f"예상하지 못한 응답 형식: {type(payload).__name__}")
    return payload

def show(rows: list[dict[str, Any]], limit: int = 30):
    sample = rows[:limit]
    print(f"전체 {len(rows):,}건 (최대 {limit:,}건 표시)")
    if pd is not None:
        return pd.DataFrame(sample)
    return sample

def load_sample(path: str) -> list[dict[str, Any]]:
    sample_path = ROOT / path
    return json.loads(sample_path.read_text(encoding="utf-8")) if sample_path.exists() else []

## 3. 최상위 주제 확인

`parentListId="A"`는 국내통계 주제별 보기의 인구 영역입니다. 반환 행에 `LIST_ID`가 있으면 하위 목록, `TBL_ID`가 있으면 조회 가능한 통계표입니다.

In [6]:
def fetch_catalog(parent_id: str = "A", view_code: str = "MT_ZTITLE") -> list[dict[str, Any]]:
    return request_json(
        LIST_API_URL,
        {"method": "getList", "vwCd": view_code, "parentListId": parent_id},
    )

topics = fetch_catalog("A") if API_KEY else load_sample("output/kosis-list.json")
show(topics)

전체 11건 (최대 30건 표시)


,LIST_NM,LIST_ID,VW_NM,VW_CD
0,인구총조사,A_4,국내통계 주제별,MT_ZTITLE
1,주민등록인구현황,A_7,국내통계 주제별,MT_ZTITLE
2,인구동향조사,A_3,국내통계 주제별,MT_ZTITLE
3,장래인구추계,A_6,국내통계 주제별,MT_ZTITLE
4,국내인구이동통계,A_1,국내통계 주제별,MT_ZTITLE
5,출입국자및체류외국인통계,A_9,국내통계 주제별,MT_ZTITLE
6,지방자치단체외국인주민현황,A_8,국내통계 주제별,MT_ZTITLE
7,장래가구추계,A_5,국내통계 주제별,MT_ZTITLE
8,국제인구이동통계,A_2,국내통계 주제별,MT_ZTITLE
9,고령친화용품제조업실태조사,A_001,국내통계 주제별,MT_ZTITLE


## 4. 하위 목록을 따라가며 통계표 찾기

위 결과에서 관심 있는 `LIST_ID`를 `PARENT_ID`에 넣으세요. 예시는 주민등록인구현황(`A_7`)입니다.

In [7]:
PARENT_ID = "A_7"  # 관심 목록의 LIST_ID로 변경
children = fetch_catalog(PARENT_ID) if API_KEY else load_sample("output/kosis-resident-population.json")
show(children, limit=50)

전체 9건 (최대 50건 표시)


,STAT_ID,SEND_DE,TBL_ID,REC_TBL_SE,ORG_ID,VW_NM,TBL_NM,VW_CD
0,2008001,2026-07-02,DT_1B040B3,N,101,국내통계 주제별,행정구역(시군구)별 주민등록세대수,MT_ZTITLE
1,2008001,2026-07-02,DT_1B040A3,N,101,국내통계 주제별,"행정구역(시군구)별, 성별 인구수",MT_ZTITLE
2,2008001,2026-07-02,DT_1B04006,N,101,국내통계 주제별,행정구역(시군구)별/1세별 주민등록인구,MT_ZTITLE
3,2008001,2026-07-02,DT_1B04005N,N,101,국내통계 주제별,행정구역(읍면동)별/5세별 주민등록인구(2011년~),MT_ZTITLE
4,2008001,2016-07-25,DT_1B04005,N,101,국내통계 주제별,행정구역(읍면동)별/5세별 주민등록인구,MT_ZTITLE
5,1976003,2026-01-09,DT_1B040M1_1,N,101,국내통계 주제별,시군구/성/연령(1세)별 주민등록연앙인구(2023~),MT_ZTITLE
6,1976003,2026-01-09,DT_1B040M5_1,N,101,국내통계 주제별,시군구/성/연령(5세)별 주민등록연앙인구(2023~),MT_ZTITLE
7,1976003,2024-01-08,DT_1B040M1,N,101,국내통계 주제별,시군구/성/연령(1세)별 주민등록연앙인구,MT_ZTITLE
8,1976003,2024-01-08,DT_1B040M5,N,101,국내통계 주제별,시군구/성/연령(5세)별 주민등록연앙인구,MT_ZTITLE


### 키워드로 통계표 검색

현재 조회한 목록에서 통계표명(`TBL_NM`) 또는 목록명(`LIST_NM`)을 검색합니다.

In [8]:
KEYWORD = "성별"
matched = [
    row for row in children
    if KEYWORD.casefold() in str(row.get("TBL_NM") or row.get("LIST_NM") or "").casefold()
]
show(matched, limit=100)

전체 1건 (최대 100건 표시)


,STAT_ID,SEND_DE,TBL_ID,REC_TBL_SE,ORG_ID,VW_NM,TBL_NM,VW_CD
0,2008001,2026-07-02,DT_1B040A3,N,101,국내통계 주제별,"행정구역(시군구)별, 성별 인구수",MT_ZTITLE


## 5. 여러 단계 자동 탐색

목록 트리를 지정 깊이까지 순회합니다. API 호출 수가 빠르게 늘 수 있으므로 처음에는 `MAX_DEPTH=1`로 확인하세요.

In [9]:
def walk_catalog(parent_id: str, max_depth: int = 1, view_code: str = "MT_ZTITLE") -> list[dict[str, Any]]:
    result: list[dict[str, Any]] = []

    def visit(current_id: str, depth: int) -> None:
        rows = fetch_catalog(current_id, view_code)
        for row in rows:
            result.append({**row, "_PARENT_ID": current_id, "_DEPTH": depth})
            child_id = row.get("LIST_ID")
            if child_id and depth < max_depth:
                visit(str(child_id), depth + 1)

    visit(parent_id, 0)
    return result

MAX_DEPTH = 1
tree_rows = walk_catalog("A_7", MAX_DEPTH) if API_KEY else children
tables = [row for row in tree_rows if row.get("TBL_ID")]
show(tables, limit=100)

전체 9건 (최대 100건 표시)


,STAT_ID,SEND_DE,TBL_ID,REC_TBL_SE,ORG_ID,VW_NM,TBL_NM,VW_CD,_PARENT_ID,_DEPTH
0,2008001,2026-07-02,DT_1B040B3,N,101,국내통계 주제별,행정구역(시군구)별 주민등록세대수,MT_ZTITLE,A_7,0
1,2008001,2026-07-02,DT_1B040A3,N,101,국내통계 주제별,"행정구역(시군구)별, 성별 인구수",MT_ZTITLE,A_7,0
2,2008001,2026-07-02,DT_1B04006,N,101,국내통계 주제별,행정구역(시군구)별/1세별 주민등록인구,MT_ZTITLE,A_7,0
3,2008001,2026-07-02,DT_1B04005N,N,101,국내통계 주제별,행정구역(읍면동)별/5세별 주민등록인구(2011년~),MT_ZTITLE,A_7,0
4,2008001,2016-07-25,DT_1B04005,N,101,국내통계 주제별,행정구역(읍면동)별/5세별 주민등록인구,MT_ZTITLE,A_7,0
5,1976003,2026-01-09,DT_1B040M1_1,N,101,국내통계 주제별,시군구/성/연령(1세)별 주민등록연앙인구(2023~),MT_ZTITLE,A_7,0
6,1976003,2026-01-09,DT_1B040M5_1,N,101,국내통계 주제별,시군구/성/연령(5세)별 주민등록연앙인구(2023~),MT_ZTITLE,A_7,0
7,1976003,2024-01-08,DT_1B040M1,N,101,국내통계 주제별,시군구/성/연령(1세)별 주민등록연앙인구,MT_ZTITLE,A_7,0
8,1976003,2024-01-08,DT_1B040M5,N,101,국내통계 주제별,시군구/성/연령(5세)별 주민등록연앙인구,MT_ZTITLE,A_7,0


## 6. 선택한 통계표의 실제 데이터 요청

통계자료 API는 표마다 항목·분류 코드가 다릅니다. 아래 예시는 주민등록 인구 표의 요청 틀이며, 오류가 나거나 0건이면 KOSIS 통계표 화면에서 원하는 항목/분류 코드를 확인해 `itmId`, `objL1` 등을 수정하세요. `+`는 해당 차원의 전체 항목을 뜻합니다.

In [10]:
def fetch_statistics_data(
    org_id: str,
    table_id: str,
    start_period: str,
    end_period: str,
    period_type: str = "Y",
    item_id: str = "T1+",
    object_codes: dict[str, str] | None = None,
) -> list[dict[str, Any]]:
    params = {
        "method": "getList",
        "orgId": org_id,
        "tblId": table_id,
        "itmId": item_id,
        "prdSe": period_type,
        "startPrdDe": start_period,
        "endPrdDe": end_period,
    }
    params.update(object_codes or {"objL1": "00+", "objL2": "+"})
    return request_json(DATA_API_URL, params)

# 아래 값을 선택한 표에 맞게 수정한 뒤 실행하세요.
SELECTED_ORG_ID = "101"
SELECTED_TABLE_ID = "DT_1B040A3"
START_PERIOD = "2024"
END_PERIOD = "2024"

if API_KEY:
    try:
        data_rows = fetch_statistics_data(
            SELECTED_ORG_ID, SELECTED_TABLE_ID, START_PERIOD, END_PERIOD
        )
        display(show(data_rows, limit=100))
    except RuntimeError as exc:
        print(exc)
        print("표의 항목·분류·주기 코드를 확인한 뒤 요청값을 수정하세요.")
else:
    print("API 키를 설정한 뒤 이 셀을 다시 실행하세요.")

KOSIS API 오류: 잘못된 요청 변수를 호출 하였습니다.
표의 항목·분류·주기 코드를 확인한 뒤 요청값을 수정하세요.


## 7. 탐색 결과 저장

In [ ]:
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

json_path = OUTPUT_DIR / "kosis_catalog_explored.json"
json_path.write_text(json.dumps(tree_rows, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"저장 완료: {json_path.resolve()}")

if pd is not None:
    csv_path = OUTPUT_DIR / "kosis_catalog_explored.csv"
    pd.DataFrame(tree_rows).to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"저장 완료: {csv_path.resolve()}")

## 응답 필드 읽는 법

- `LIST_ID`, `LIST_NM`: 더 탐색할 수 있는 목록 ID와 이름
- `TBL_ID`, `TBL_NM`: 실제 데이터 조회에 사용하는 통계표 ID와 이름
- `ORG_ID`: 통계 작성기관 ID
- `STAT_ID`: 통계조사 ID
- `PRD_DE`, `PRD_SE`: 자료의 시점과 주기
- `ITM_ID`, `ITM_NM`: 통계 항목 코드와 이름
- `C1`/`C1_NM`, `C2`/`C2_NM` 등: 분류 코드와 이름
- `DT`: 통계값